# 02. Создание эмбеддингов и их сохранение

Подготовленные данные преобразуем в векторные представления и сохраним их.

*First run the following cells and restart Google Colab session if prompted. Then run the notebook again cell by cell to explore the code.*

In [ ]:
import deeplake

Mount a drive or implement the method that best fits your project to retrieve API tokens.

In [ ]:
#The OpenAI KeyActiveloop and OpenAI API keys
import os
import openai

os.environ['GROQ_API_KEY']="YOUR_GROQ_API_KEY"

ACTIVELOOP_TOKEN=API_token
os.environ['ACTIVELOOP_TOKEN'] =ACTIVELOOP_TOKEN

Chunking the data

In [ ]:
source_text = "llm.rus.txt"
with open(source_text, 'r') as f:
    text = f.read()

CHUNK_SIZE = 1000
chunked_text = [text[i:i+CHUNK_SIZE] for i in range(0,len(text), CHUNK_SIZE)]

## Verifying if the vector store exists or create it

If vector store doesn't exist, the following code will create it and display a message.

If the vectore store exists, only the "Vector store exists" message will be displayed.

Deep Lake

**Replace `hub://denis76/space_exploration_v1` by your organization and dataset name**

In [10]:
vector_store_path = "hub://denis76/space_exploration_v1"

In [ ]:
from deeplake.core.vectorstore.deeplake_vectorstore import VectorStore
import deeplake.util

try:
    # Attempt to load the vector store
    vector_store = VectorStore(path=vector_store_path)
    print("Vector store exists")
except FileNotFoundError:
    print("Vector store does not exist. You can create it.")
    # Code to create the vector store goes here
    create_vector_store=True


## The embedding function

In [12]:
def embedding_function(texts, model="text-embedding-3-small"):
   if isinstance(texts, str):
       texts = [texts]
   texts = [t.replace("\n", " ") for t in texts]
   return [data.embedding for data in openai.embeddings.create(input = texts, model=model).data]

## Adding data to the vector store

In [13]:
add_to_vector_store=True

In [ ]:
if add_to_vector_store == True:
    with open(source_text, 'r') as f:
        text = f.read()
        CHUNK_SIZE = 1000
        chunked_text = [text[i:i+1000] for i in range(0, len(text), CHUNK_SIZE)]

vector_store.add(text = chunked_text,
              embedding_function = embedding_function,
              embedding_data = chunked_text,
              metadata = [{"source": source_text}]*len(chunked_text))


# Vector Store information

Summary

In [ ]:
# Print the summary of the Vector Store
print(vector_store.summary())

Visualize

Online:
https://app.activeloop.ai/datasets/mydatasets/

In [ ]:
ds = deeplake.load(vector_store_path)

Dataset size

In [17]:
#Estimates the size in bytes of the dataset.
ds_size=ds.size_approx()

In [18]:
# Convert bytes to megabytes and limit to 5 decimal places
ds_size_mb = ds_size / 1048576
print(f"Dataset size in megabytes: {ds_size_mb:.5f} MB")

# Convert bytes to gigabytes and limit to 5 decimal places
ds_size_gb = ds_size / 1073741824
print(f"Dataset size in gigabytes: {ds_size_gb:.5f} GB")

Dataset size in megabytes: 55.31311 MB
Dataset size in gigabytes: 0.05402 GB
